# ContextCheck: Colab GPU training

This notebook runs the project's real SNLI preprocessing, baseline, and DistilBERT training scripts. In Colab, choose **Runtime → Change runtime type → T4 GPU** (or another GPU). Upload a ZIP of the project source (exclude model weights, caches, and downloaded data). Set `TRAIN_LIMIT` to `None` for full training or a smaller integer for a repeatable pilot run. Test examples are reserved for final evaluation.

In [ ]:
from google.colab import files
from pathlib import Path
import os
import zipfile

uploaded = files.upload()  # Select a ZIP containing app.py, requirements.txt, and src/.
zip_name = next(name for name in uploaded if name.lower().endswith('.zip'))
project_root = Path('/content/contextcheck')
project_root.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_name) as archive:
    archive.extractall(project_root)
candidates = [path for path in project_root.rglob('requirements.txt') if (path.parent / 'src').is_dir()]
if not candidates:
    raise FileNotFoundError('The uploaded ZIP must contain the ContextCheck project source.')
os.chdir(candidates[0].parent)
print('Project directory:', Path.cwd())

## Inspect and preprocess SNLI

The loader prints the feature structure and row counts before and after removing unlabeled or unusable examples, while preserving the original split boundaries.

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
%pip install -q --upgrade "protobuf>=5.26,<6"

In [ ]:
import torch
import transformers
import datasets
import streamlit

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("streamlit:", streamlit.__version__)
print("GPU available:", torch.cuda.is_available())

In [ ]:
import os
import sys

os.chdir("/content/contextcheck")
sys.path.insert(0, os.getcwd())

print("Current directory:", os.getcwd())
print("Project files:", os.listdir(".")[:10])

In [ ]:
import torch
import transformers
import datasets
import streamlit
import google.protobuf

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("streamlit:", streamlit.__version__)
print("protobuf:", google.protobuf.__version__)
print("GPU available:", torch.cuda.is_available())

In [ ]:
from src.dataset_loader import load_snli
snli, preprocessing_stats = load_snli()
preprocessing_stats

In [ ]:
import shutil
import subprocess

python_executable = shutil.which("python")
print("Using Python:", python_executable)

subprocess.run([
    python_executable,
    "-m",
    "pip",
    "install",
    "--upgrade",
    "protobuf>=5.26,<6"
], check=True)

In [ ]:
import os
os.chdir("/content/contextcheck")
print(os.getcwd())

In [ ]:
import shutil
import subprocess

python_executable = shutil.which("python")

subprocess.run([
    python_executable,
    "-m",
    "pip",
    "install",
    "--upgrade",
    "protobuf>=6.31.1,<7"
], check=True)

In [ ]:
import shutil
import subprocess

python_executable = shutil.which("python")
print("Using Python:", python_executable)

if python_executable is None:
    raise RuntimeError("Python executable not found.")

command = [
    python_executable,
    "-m",
    "src.train_transformer",
    "--epochs", "2",
    "--batch-size", "8",
    "--gradient-accumulation-steps", "2",
]

result = subprocess.run(
    command,
    text=True,
    capture_output=True
)

print("RETURN CODE:", result.returncode)
print(result.stdout[-12000:])
print(result.stderr[-12000:])

In [ ]:
import subprocess
import sys

command = [
    sys.executable,
    "-m",
    "src.train_transformer",
    "--epochs", "2",
    "--batch-size", "8",
    "--gradient-accumulation-steps", "2",
]

result = subprocess.run(
    command,
    text=True,
    capture_output=True,
)

print("RETURN CODE:", result.returncode)
print("\n--- LAST OUTPUT ---")
print(result.stdout[-12000:])
print("\n--- ERROR DETAILS ---")
print(result.stderr[-12000:])

In [ ]:
from pathlib import Path

model_dir = Path("/content/contextcheck/models/contextcheck_distilbert")
evaluation_file = Path("/content/contextcheck/artifacts/evaluation_results.json")

print("Model folder:", model_dir.exists())
print("Evaluation file:", evaluation_file.exists())
print("Model weight:", (model_dir / "model.safetensors").exists())
print("Config:", (model_dir / "config.json").exists())
print("Tokenizer:", (model_dir / "tokenizer.json").exists())

In [ ]:
print("\n".join(result.stderr.splitlines()[-35:]))

## Save artifacts to Google Drive

Model weights can be large. This optional cell copies the fine-tuned model and measured reports to Drive so the Streamlit app can load them locally later. Do not commit the weights to Git.

In [ ]:
import os
from pathlib import Path

folders = [
    Path("/content/contextcheck/models/contextcheck_distilbert"),
    Path("/content/contextcheck/models/baseline"),
    Path("/content/contextcheck/artifacts"),
]

for folder in folders:
    print(folder, folder.exists())

In [ ]:
import shutil
from pathlib import Path

for folder in (
    "models/contextcheck_distilbert",
    "models/baseline",
    "artifacts",
):
    path = Path("/content/contextcheck") / folder
    if path.exists():
        archive = shutil.make_archive(
            f"/content/{folder.replace('/', '_')}",
            "zip",
            path,
        )
        print("Created:", archive)

In [ ]:
from google.colab import files
files.download("/content/models_contextcheck_distilbert.zip")

In [ ]:
from google.colab import files
files.download("/content/artifacts.zip")